# Assignment (Bonus): Build Your MapReduce (5 marks)

**CIS 41 03 - Big Data Analytics - Lecture 06: Processing Big Data: Hadoop & MapReduce**

Student: **Mahmoud Khaled Mahmoud Yassin Badr** - ID: **231005511**

### The task

Write a small **MapReduce-like engine** in any language you like. No Hadoop, no Spark: you build the phases yourself.
This notebook builds it in plain Python.

1. **Input**: a folder of text files. Each file is one **split**.
2. **Map**: for each split, emit `(word, 1)` pairs.
3. **Run the mappers at the same time** (threads or processes), one per split.
4. **Combine + Partition**: local subtotals, then send each word to one of **2 reducers**.
5. **Shuffle & Sort**, then **Reduce**: sum the values for each word and write one output file per reducer.

### The one rule

> The grouping and counting must be **your own map / shuffle / reduce code**.
> So: no `collections.Counter`, no `pandas.groupby`, no `itertools.groupby`. Plain dicts and loops only.

### Marking scheme

| Part | Marks |
|---|---|
| Map emits correct `(key, value)` pairs | 1 |
| Mappers run in parallel on the splits | 1 |
| Combiner + partitioner to 2 reducers | 1 |
| Shuffle & sort + reduce give correct totals | 1 |
| README: how to run it, sample input/output, one screenshot | 1 |
| **Total** | **5** |

### Self-check: test it on the 3 ETI claim notes

```
car 3    damage 3    fire 1    roof 1    theft 1    water 3
```

Same numbers as the lecture walk-through and Lab 06.

## 0. The idea in one line

> **"Don't move the data to the code. Move the code to the data."**

In Hadoop, **HDFS** stores the data in blocks and **MapReduce** sends the code to the node that already
holds the block (**data locality**). We have no cluster here, so we fake it:

| Real Hadoop | Our notebook |
|---|---|
| HDFS block (128 MB) on a DataNode | one text file in the `claims/` folder |
| map task on the node holding the block | one Python thread reading that one file |
| network between map and reduce | a Python list of pairs |
| `part-00000`, `part-00001` output files | real files in the `output/` folder |

The **five phases** of one MapReduce job stay exactly the same:

**Map -> Combine -> Partition -> Shuffle & Sort -> Reduce**

* Map task = map -> combine -> partition
* Reduce task = shuffle & sort -> reduce

In [1]:
import os, re, shutil, time, threading
from concurrent.futures import ThreadPoolExecutor

BASE    = "mapreduce_job"
INPUT   = os.path.join(BASE, "claims")    # folder of text files = our "HDFS"
OUTPUT  = os.path.join(BASE, "output")    # part-00000, part-00001

# start clean every run
if os.path.isdir(BASE):
    shutil.rmtree(BASE)
os.makedirs(INPUT)
os.makedirs(OUTPUT)

NUM_REDUCERS = 2
print("ready")

ready


## 1. Input: a folder of text files, each file is one split

The **Map** phase cuts the input file into smaller pieces called **splits**, and **each split goes to one
mapper** (one mapper per split).

Here we skip the cutting: we simply put **one claim note in one file**, so the folder already *is* the list
of splits. These are the same 3 ETI claim notes from the lecture walk-through:

* `"car fire car damage"`
* `"water roof water damage"`
* `"car theft water damage"`

12 words in total, so the 6 counts at the end must add up to 12.

In [2]:
claim_notes = {
    "note_01.txt": "car fire car damage",
    "note_02.txt": "water roof water damage",
    "note_03.txt": "car theft water damage",
}

for name, text in claim_notes.items():
    with open(os.path.join(INPUT, name), "w") as f:
        f.write(text + "\n")

# the splits = the files in the folder
splits = sorted(os.path.join(INPUT, n) for n in os.listdir(INPUT) if n.endswith(".txt"))

print("splits (one mapper each):", len(splits))
for s in splits:
    print(" ", s, "->", open(s).read().strip())

splits (one mapper each): 3
  mapreduce_job/claims/note_01.txt -> car fire car damage
  mapreduce_job/claims/note_02.txt -> water roof water damage
  mapreduce_job/claims/note_03.txt -> car theft water damage


## 2. Phase 1 - Map: emit `(word, 1)` for every word

The mapper **runs once for every record** and applies your own logic. Its output is a **list of key-value
pairs**, where several pairs may share the same key.

For one input record a mapper may emit **no pair** (filtering) or **many pairs** (demultiplexing). Here one
claim note of 4 words becomes 4 pairs:

```
"car fire car damage"
  -> (car, 1) (fire, 1) (car, 1) (damage, 1)
```

Note the list is **not** grouped yet: `car` appears twice. Grouping is not the mapper's job.

In [3]:
def map_function(line):
    """Map: one record in, a list of (word, 1) pairs out. Our own logic."""
    pairs = []
    for word in re.findall(r"[a-z']+", line.lower()):   # split the record into words
        pairs.append((word, 1))                         # emit (key, value)
    return pairs


# quick look at what Mapper 1 emits
print(map_function("car fire car damage"))

[('car', 1), ('fire', 1), ('car', 1), ('damage', 1)]


## 3. Phase 2 - Combine (optional): a mini-reducer on the mapper's own node

The **combiner** is a mini-reducer that runs on the **mapper's own node**. It summarizes a mapper's output
**before it travels over the network**, so less data is moved between map and reduce.

```
(car,1) (fire,1) (car,1) (damage,1)     4 pairs
      -> combine on the same node ->
(car,2) (fire,1) (damage,1)             3 pairs
```

**Not always safe**: works for **sum, count, max, min**, but **not for average** - a combiner sees only part
of the data, and an average of averages is wrong. Word count is a sum, so here it is safe.

It groups the values for one key (`key -> list of values`) and applies the reduce logic locally.

In [4]:
def combine_function(pairs):
    """Combine: local subtotals on the mapper's own node. Our own grouping, no Counter."""
    local = {}                      # key -> list of values
    for key, value in pairs:
        if key not in local:
            local[key] = []
        local[key].append(value)

    combined = []
    for key in local:               # apply the reduce logic locally: sum
        total = 0
        for v in local[key]:
            total += v
        combined.append((key, total))
    return combined


print("before combine:", map_function("car fire car damage"))
print("after  combine:", combine_function(map_function("car fire car damage")))

before combine: [('car', 1), ('fire', 1), ('car', 1), ('damage', 1)]
after  combine: [('car', 2), ('fire', 1), ('damage', 1)]


## 4. Phase 3 - Partition: one partition per reducer

The **partitioner** runs only when there is **more than one reducer**. It splits the map (or combine) output
into **partitions, one per reducer**, and returns the **reducer number** for each record.

* number of partitions = number of reducers (here: **2**)
* **all records with the same key go to the same partition**, so to the same reducer

We use the same rule as the lecture: **Reducer 1 = words a-f**, **Reducer 2 = words g-z**.

A **skewed key** (one word far more common than the rest) would make one reducer finish late, which is why a
real job can customize the partitioner.

In [5]:
def partition_label(reducer_id, num_reducers=NUM_REDUCERS):
    """The word range a reducer is responsible for, for printing only."""
    if num_reducers == 2:
        return "a-f" if reducer_id == 0 else "g-z"
    return f"hash % {num_reducers} == {reducer_id}"


def partition_function(key, num_reducers=NUM_REDUCERS):
    """Partition: return the reducer number for this key. Same key -> same reducer, always."""
    if num_reducers == 2:
        return 0 if key[0] <= "f" else 1    # Reducer 1 = a-f, Reducer 2 = g-z
    # general fallback: hash the key, so equal keys still land on one reducer
    return sum(ord(c) for c in key) % num_reducers


for w in ["car", "damage", "fire", "roof", "theft", "water"]:
    print(f"{w:8s} -> Reducer {partition_function(w) + 1}")

car      -> Reducer 1
damage   -> Reducer 1
fire     -> Reducer 1
roof     -> Reducer 2
theft    -> Reducer 2
water    -> Reducer 2


## 5. Phase 1-3 together: the map task, run in parallel

> **Map task = map -> combine -> partition.**

Now we run **one mapper per split, all at the same time** (threads), exactly like a cluster starts one map
task per block. Each mapper returns its **partitions**: a small box of pairs for every reducer.

Each mapper prints its thread name, then the cell shows the wall time and the totals the reducers will write.

In [6]:
print_lock = threading.Lock()    # so two threads cannot mix their printed lines


def map_task(split_path):
    """One map task on one split: map -> combine -> partition."""

    # 1 - Map: read the split record by record, emit (word, 1)
    pairs = []
    for line in open(split_path):
        pairs += map_function(line)

    time.sleep(0.3)        # pretend this split is big, so the overlap is visible

    # 2 - Combine: local subtotals, on this mapper's own node
    combined = combine_function(pairs)

    # 3 - Partition: one box per reducer
    partitions = [[], []]
    for key, value in combined:
        r = partition_function(key)
        partitions[r].append((key, value))

    name = os.path.basename(split_path)
    thread = threading.current_thread().name
    with print_lock:
        print(f"[{thread}] {name}: map {len(pairs)} pairs -> combine {len(combined)} pairs")
    return partitions


start = time.time()
with ThreadPoolExecutor(max_workers=3) as pool:      # 3 mappers at the same time
    all_partitions = list(pool.map(map_task, splits))
wall = time.time() - start

print()
print("3 mappers, 0.3 s of work each")
print("wall time:", round(wall, 2), "s    (0.3 = parallel, 0.9 = one after another)")

# where the job is heading: the totals the reducers will write
# (sections 6 and 7 do these two phases properly, phase by phase)
print()
print("after reduce:")
for r in range(NUM_REDUCERS):
    grouped = {}                                  # shuffle & sort: key -> list of values
    for partitions in all_partitions:
        for key, value in partitions[r]:
            if key not in grouped:
                grouped[key] = []
            grouped[key].append(value)

    for key in sorted(grouped):                   # reduce: sum each list
        total = 0
        for v in grouped[key]:
            total += v
        print(f"  Reducer {r + 1}: {key} {total}")

[ThreadPoolExecutor-0_0] note_01.txt: map 4 pairs -> combine 3 pairs
[ThreadPoolExecutor-0_1] note_02.txt: map 4 pairs -> combine 3 pairs
[ThreadPoolExecutor-0_2] note_03.txt: map 4 pairs -> combine 4 pairs

3 mappers, 0.3 s of work each
wall time: 0.31 s    (0.3 = parallel, 0.9 = one after another)

after reduce:
  Reducer 1: car 3
  Reducer 1: damage 3
  Reducer 1: fire 1
  Reducer 2: roof 1
  Reducer 2: theft 1
  Reducer 2: water 3


### What each mapper produced

This is the picture from the lecture slide "Word Count, Step by Step":

```
Mapper 1  "car fire car damage"     -> (car,2) (fire,1) (damage,1)  |
Mapper 2  "water roof water damage" -> (damage,1)  |  (water,2) (roof,1)
Mapper 3  "car theft water damage"  -> (car,1) (damage,1)  |  (theft,1) (water,1)
```

The `|` is the **partition** line: left of it goes to Reducer 1 (a-f), right of it to Reducer 2 (g-z).

In [7]:
for i, partitions in enumerate(all_partitions, start=1):
    left  = "  ".join(f"({k}, {v})" for k, v in partitions[0])
    right = "  ".join(f"({k}, {v})" for k, v in partitions[1])
    print(f"Mapper {i}: {left}   |   {right}")

Mapper 1: (car, 2)  (fire, 1)  (damage, 1)   |   
Mapper 2: (damage, 1)   |   (water, 2)  (roof, 1)
Mapper 3: (car, 1)  (damage, 1)   |   (theft, 1)  (water, 1)


## 6. Phase 4 - Shuffle & Sort

* **Shuffle**: the partitions from **all** mappers are copied **over the network** to the reducer nodes.
  Each reducer may receive the same key many times, from different mappers.
* **Sort**: the pairs are grouped and sorted by key, then merged into **one pair per key:
  `key -> list of all its values`**.

```
Reducer 1 receives:  (car,2) (fire,1) (damage,1) (damage,1) (car,1) (damage,1)
     -> sort by key, group the values ->
car -> [2, 1]     damage -> [1, 1, 1]     fire -> [1]
```

This is the **only phase that crosses the network in bulk**, so in a real cluster it is usually the slowest.

In [8]:
def shuffle_and_sort(all_partitions, reducer_id):
    """Shuffle: collect this reducer's partition from every mapper. Sort: group by key, sorted."""
    # shuffle - everything addressed to this reducer arrives here
    received = []
    for partitions in all_partitions:
        received += partitions[reducer_id]

    # sort + group -> key -> list of values  (our own grouping, no itertools.groupby)
    grouped = {}
    for key, value in received:
        if key not in grouped:
            grouped[key] = []
        grouped[key].append(value)

    return received, {k: grouped[k] for k in sorted(grouped)}   # keys in sorted order


shuffled = {}
for r in range(NUM_REDUCERS):
    received, grouped = shuffle_and_sort(all_partitions, r)
    shuffled[r] = grouped
    print(f"Reducer {r+1} received over the network: " +
          "  ".join(f"({k}, {v})" for k, v in received))
    print(f"Reducer {r+1} after sort & group:        " +
          "     ".join(f"{k} -> {v}" for k, v in grouped.items()))
    print()

Reducer 1 received over the network: (car, 2)  (fire, 1)  (damage, 1)  (damage, 1)  (car, 1)  (damage, 1)
Reducer 1 after sort & group:        car -> [2, 1]     damage -> [1, 1, 1]     fire -> [1]

Reducer 2 received over the network: (water, 2)  (roof, 1)  (theft, 1)  (water, 1)
Reducer 2 after sort & group:        roof -> [1]     theft -> [1]     water -> [2, 1]



## 7. Phase 5 - Reduce: one output file per reducer

For each key the reducer works through its **list of values** and writes a new key-value pair. Here the logic
is **sum the list**:

```
car    -> [2, 1]      -> car 3
damage -> [1, 1, 1]   -> damage 3
water  -> [2, 1]      -> water 3
```

**Output: one file per reducer** (`part-00000`, `part-00001`, ...), which together form the full result.

In [9]:
def reduce_function(key, values):
    """Reduce: our own logic. Sum the list of values for one key."""
    total = 0
    for v in values:
        total += v
    return (key, total)


def reduce_task(reducer_id, grouped, output_dir=OUTPUT):
    """One reduce task: reduce every key, write part-0000N."""
    path = os.path.join(output_dir, f"part-{reducer_id:05d}")
    with open(path, "w") as f:
        for key in grouped:                               # already sorted
            k, total = reduce_function(key, grouped[key])
            f.write(f"{k}\t{total}\n")
    return path


# reducers can run at the same time too
with ThreadPoolExecutor(max_workers=NUM_REDUCERS) as pool:
    out_files = list(pool.map(lambda r: reduce_task(r, shuffled[r]), range(NUM_REDUCERS)))

for r, path in enumerate(out_files):
    span = "a-f" if r == 0 else "g-z"
    print(f"--- {path}   (Reducer {r + 1}, words {span}) ---")
    print(open(path).read(), end="")

--- mapreduce_job/output/part-00000   (Reducer 1, words a-f) ---
car	3
damage	3
fire	1
--- mapreduce_job/output/part-00001   (Reducer 2, words g-z) ---
roof	1
theft	1
water	3


## 8. Self-check

The expected result from the lecture walk-through and Lab 06:

| word | count | reducer |
|---|---|---|
| car | 3 | R1 (a-f) |
| damage | 3 | R1 (a-f) |
| fire | 1 | R1 (a-f) |
| roof | 1 | R2 (g-z) |
| theft | 1 | R2 (g-z) |
| water | 3 | R2 (g-z) |

`3 + 3 + 1 + 1 + 1 + 3 = 12`, exactly the 12 words we started with.

Computed by **our own map, shuffle and reduce functions, not by `Counter`**.

In [10]:
expected = {"car": 3, "damage": 3, "fire": 1, "roof": 1, "theft": 1, "water": 3}

# read the result back out of the part files, like a real job would
result = {}
for name in sorted(os.listdir(OUTPUT)):
    for line in open(os.path.join(OUTPUT, name)):
        word, count = line.split("\t")
        result[word] = int(count)

print(f"{'word':10s} {'count':>5s}  reducer   expected")
for word in sorted(result):
    r = partition_function(word) + 1
    span = "a-f" if r == 1 else "g-z"
    ok = "OK" if result[word] == expected.get(word) else "WRONG"
    print(f"{word:10s} {result[word]:5d}  R{r} ({span})  {ok}")

total_words = 0
for w in result:
    total_words += result[w]

print()
print("sum of counts :", total_words, "(must be 12)")
print("matches lecture:", result == expected)
assert result == expected, "counts do not match the lecture walk-through"
assert total_words == 12
print("\nSelf-check passed.")

word       count  reducer   expected
car            3  R1 (a-f)  OK
damage         3  R1 (a-f)  OK
fire           1  R1 (a-f)  OK
roof           1  R2 (g-z)  OK
theft          1  R2 (g-z)  OK
water          3  R2 (g-z)  OK

sum of counts : 12 (must be 12)
matches lecture: True

Self-check passed.


## 9. One job, one function

Everything above, wrapped as one callable **MapReduce job**, so it runs on any folder of text files with any
number of reducers. This is the part you hand in as the engine.

Nothing new is written here: it only calls the five functions already built -
`map_function`, `combine_function`, `partition_function`, `shuffle_and_sort`, `reduce_task`.

In [11]:
def mapreduce(input_dir, output_dir, num_reducers=2, use_combiner=True):
    """The whole job in one function, built only from the functions above."""
    os.makedirs(output_dir, exist_ok=True)
    splits = sorted(os.path.join(input_dir, n)
                    for n in os.listdir(input_dir) if n.endswith(".txt"))

    def one_map_task(split_path):
        pairs = []
        for line in open(split_path):
            pairs += map_function(line)                 # 1 - Map
        if use_combiner:
            pairs = combine_function(pairs)             # 2 - Combine
        parts = [[] for _ in range(num_reducers)]       # 3 - Partition
        for key, value in pairs:
            parts[partition_function(key, num_reducers)].append((key, value))
        return parts

    # map task: one per split, all at the same time
    with ThreadPoolExecutor(max_workers=max(1, len(splits))) as pool:
        all_parts = list(pool.map(one_map_task, splits))

    def one_reduce_task(r):
        received, grouped = shuffle_and_sort(all_parts, r)    # 4 - Shuffle & Sort
        return reduce_task(r, grouped, output_dir)            # 5 - Reduce

    # reduce task: one per reducer, at the same time, same as the mappers
    with ThreadPoolExecutor(max_workers=num_reducers) as pool:
        return list(pool.map(one_reduce_task, range(num_reducers)))


# writes into the same output/ folder: one job, one place for the results
files = mapreduce(INPUT, OUTPUT, num_reducers=2)
for r, p in enumerate(files):
    print(f"--- {p}   (Reducer {r + 1}, words {partition_label(r)}) ---")
    print(open(p).read(), end="")

--- mapreduce_job/output/part-00000   (Reducer 1, words a-f) ---
car	3
damage	3
fire	1
--- mapreduce_job/output/part-00001   (Reducer 2, words g-z) ---
roof	1
theft	1
water	3


## 10. Same answer as SQL's `GROUP BY`

> **"GROUP BY never went away."** MapReduce is the same aggregation, written out by hand so it can run on a
> hundred machines.

| SQL | MapReduce phase |
|---|---|
| `SELECT word` | **1 - Map**: emit `(word, 1)` |
| `GROUP BY word` | **4 - Shuffle & Sort**: bring each word's values together |
| `SUM(1)` | **5 - Reduce**: add up each word's list |

One machine writes the plan for you; with MapReduce **you** write the plan, and it scales past any single
disk. The cell below proves both give the same counts.

In [12]:
import sqlite3

con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE words (word TEXT)")
for path in splits:
    for line in open(path):
        for w in re.findall(r"[a-z']+", line.lower()):
            con.execute("INSERT INTO words VALUES (?)", (w,))

sql_rows = con.execute(
    "SELECT word, COUNT(*) FROM words GROUP BY word ORDER BY word"
).fetchall()

print(f"{'word':10s} {'SQL':>4s} {'MapReduce':>10s}")
for word, n in sql_rows:
    print(f"{word:10s} {n:4d} {result[word]:10d}")

print()
print("same answer:", {w: n for w, n in sql_rows} == result)

word        SQL  MapReduce
car           3          3
damage        3          3
fire          1          1
roof          1          1
theft         1          1
water         3          3

same answer: True


## 11. README

The last mark is the **README: how to run it, sample input/output, one screenshot**. The cell below writes
`README.md` next to the job. **You still have to add the screenshot yourself**: run this notebook, take one
picture of the output of the self-check cell, save it as `screenshot.png` in the same folder, and it is
already linked at the bottom of the README.

In [13]:
readme = """# Bonus Assignment - Build Your MapReduce (5 marks)

**CIS 41 03 - Big Data Analytics - Lecture 06: Processing Big Data: Hadoop & MapReduce**

Student: **Mahmoud Khaled Mahmoud Yassin Badr** - ID: **231005511**

A small **MapReduce-like engine** written from scratch in plain Python.
**No Hadoop, no Spark**: the five phases are built by hand.

**Map -> Combine -> Partition -> Shuffle & Sort -> Reduce**

## 1. How to run

```bash
jupyter notebook bonus_mapreduce.ipynb    # then: Cell -> Run All
```

or without opening it:

```bash
jupyter nbconvert --to notebook --execute --inplace bonus_mapreduce.ipynb
```

Nothing to install: **standard library only** (`os`, `re`, `shutil`, `time`, `threading`,
`concurrent.futures`, `sqlite3`). Runs the same in Google Colab.

### The Streamlit UI (extra)

The same engine also runs behind a small web UI, which shows every phase on its own tab:

```bash
pip install streamlit
streamlit run app.py
```

| File | What it is |
|---|---|
| `bonus_mapreduce.ipynb` | the assignment: the five phases, built and explained one at a time |
| `mapreduce_engine.py` | the same five functions in one importable file |
| `app.py` | the Streamlit UI on top of `mapreduce_engine.py` |

The UI lets you change the number of reducers, switch the combiner off, upload your own `.txt`
splits, and download the `part-0000N` files.

Running it creates this folder next to the notebook:

```
mapreduce_job/
  claims/       the input splits (one text file = one split)
  output/       part-00000, part-00001   (one file per reducer)
```

## 2. Where each mark is proved

| # | Marking scheme | Where it is in the notebook | Evidence |
|---|---|---|---|
| 1 | Map emits correct `(key, value)` pairs | section 2, `map_function` | `"car fire car damage"` -> `(car,1) (fire,1) (car,1) (damage,1)` |
| 2 | Mappers run in parallel on the splits | section 5, `map_task` + `ThreadPoolExecutor` | 3 mappers, 0.3 s each, **wall time about 0.3 s**, not 0.9 s; each mapper prints its own thread name |
| 3 | Combiner + partitioner to 2 reducers | sections 3 and 4, `combine_function`, `partition_function` | combine: 4 pairs -> 3 pairs on the mapper's own node; partition: words **a-f -> Reducer 1**, **g-z -> Reducer 2** |
| 4 | Shuffle & sort + reduce give correct totals | sections 6, 7, 8 | `car -> [2,1]`, `damage -> [1,1,1]`; the 2 reducers run in their own threads; totals match the lecture, and `assert` fails the run if they do not |
| 5 | README: how to run, sample input/output, screenshot | this file | sections 1, 3, 4 and 6 below, with `result.png` |

## 3. Sample input

A folder of text files. **Each file is one split**, and each split gets its own mapper running at the
same time in its own thread. The input is the 3 ETI claim notes from the lecture walk-through:

```
mapreduce_job/claims/note_01.txt    car fire car damage
mapreduce_job/claims/note_02.txt    water roof water damage
mapreduce_job/claims/note_03.txt    car theft water damage
```

12 words in total.

## 4. Sample output

One output file per reducer, exactly as MapReduce writes `part-00000`, `part-00001`:

```
$ cat mapreduce_job/output/part-00000     $ cat mapreduce_job/output/part-00001
car     3                                 roof    1
damage  3                                 theft   1
fire    1                                 water   3
```

`3 + 3 + 1 + 1 + 1 + 3 = 12`, exactly the 12 words of the input.

### Self-check (required numbers)

```
car 3    damage 3    fire 1    roof 1    theft 1    water 3
```

Same numbers as the lecture walk-through and Lab 06. The notebook checks them two ways:

* an `assert` against the expected dictionary (the run fails loudly if a count is wrong);
* a `sqlite3` `SELECT word, COUNT(*) ... GROUP BY word` over the same words, printed side by side -
  `same answer: True`.

## 5. The phases, and the one rule

| Phase | Function | What it does |
|---|---|---|
| 1 - Map | `map_function` | emits `(word, 1)` for every word in the record |
| 2 - Combine | `combine_function` | mini-reducer on the mapper's own node: local subtotals before the network |
| 3 - Partition | `partition_function` | one partition per reducer, 2 reducers: a-f -> R1, g-z -> R2; same key always to the same reducer |
| 4 - Shuffle & Sort | `shuffle_and_sort` | copies each reducer's partitions from every mapper, groups and sorts them `key -> list of values` |
| 5 - Reduce | `reduce_function` | sums each list, writes one output file per reducer |

* **Map task** = map -> combine -> partition. **Reduce task** = shuffle & sort -> reduce.
* The mappers are started with a `ThreadPoolExecutor`, **one worker per split**, so they run at the
  same time. The printed thread names and the wall time show the overlap.
* The **reducers run in threads too**, one worker per reducer, in section 7 and inside the engine.
* `mapreduce(input_dir, output_dir, num_reducers)` in section 9 is the whole engine in one function,
  and works on any folder of text files with any number of reducers. It writes no new logic: it only
  calls `map_function`, `combine_function`, `partition_function`, `shuffle_and_sort` and `reduce_task`.
* Threads here stand in for what a real cluster does with separate processes on separate nodes.
  Swapping `ThreadPoolExecutor` for `ProcessPoolExecutor` is a one-word change; the assignment allows
  either.

**The one rule is respected:** the grouping and counting are all own code - plain dicts and loops.
**No `collections.Counter`, no `itertools.groupby`, no `pandas`.** `sqlite3` appears only in section 10,
as an independent cross-check of the result, never to compute it.

### Why the combiner is safe here

Word count is a **sum**, and a sum can be done in pieces. A combiner is **not** safe for an **average**:
it sees only part of the data, and an average of averages is wrong.

## 6. Result

The full result of the job, every word with its count and the reducer that wrote it:

| word | count | reducer |
|---|---|---|
| car | 3 | R1 (a-f) |
| damage | 3 | R1 (a-f) |
| fire | 1 | R1 (a-f) |
| roof | 1 | R2 (g-z) |
| theft | 1 | R2 (g-z) |
| water | 3 | R2 (g-z) |

`3 + 3 + 1 + 1 + 1 + 3 = 12`, the 12 words of the input, computed by our own map, shuffle and reduce
functions - not by `Counter`.

### Screenshot

![result](result.png)

## 7. Submission

Forked `github.com/dr-ahmed-said/Big-Data-Analytics`, code + README added in
`submissions/Lecture-06/231005511/`, pull request opened before the discussion.
Individual work.
"""

path = "README.md"
with open(path, "w") as f:
    f.write(readme)

print("written:", os.path.abspath(path))
print()
print(readme)

written: /Users/mahmoudkhaled/Documents/bonus/README.md

# Bonus Assignment - Build Your MapReduce (5 marks)

**CIS 41 03 - Big Data Analytics - Lecture 06: Processing Big Data: Hadoop & MapReduce**

Student: **Mahmoud Khaled Mahmoud Yassin Badr** - ID: **231005511**

A small **MapReduce-like engine** written from scratch in plain Python.
**No Hadoop, no Spark**: the five phases are built by hand.

**Map -> Combine -> Partition -> Shuffle & Sort -> Reduce**

## 1. How to run

```bash
jupyter notebook bonus_mapreduce.ipynb    # then: Cell -> Run All
```

or without opening it:

```bash
jupyter nbconvert --to notebook --execute --inplace bonus_mapreduce.ipynb
```

Nothing to install: **standard library only** (`os`, `re`, `shutil`, `time`, `threading`,
`concurrent.futures`, `sqlite3`). Runs the same in Google Colab.

### The Streamlit UI (extra)

The same engine also runs behind a small web UI, which shows every phase on its own tab:

```bash
pip install streamlit
streamlit run app.py
``

## 12. What is in the folder now

This is what you submit: fork the course repo, put the notebook, the `mapreduce_job/` folder (code output,
README and screenshot) in `submissions/Lecture-06/your-student-id/`, and open a pull request **before the
discussion**. Individual work: be ready to explain any line.

In [14]:
for root, dirs, files in os.walk(BASE):
    depth = root.count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    for name in sorted(files):
        size = os.path.getsize(os.path.join(root, name))
        print("  " * (depth + 1) + f"{name}  ({size} bytes)")

mapreduce_job/
  output/
    part-00000  (22 bytes)
    part-00001  (23 bytes)
  claims/
    note_01.txt  (20 bytes)
    note_02.txt  (24 bytes)
    note_03.txt  (23 bytes)
